###Paths

In [0]:
from pyspark.sql import functions as F

BRONZE_PATH = "abfss://bronze@ecommercedatalakebp01.dfs.core.windows.net"
SILVER_PATH = "abfss://silver@ecommercedatalakebp01.dfs.core.windows.net"
INGESTION_DATE = "2026-09-23"

In [0]:
path_customers=f"{BRONZE_PATH}/customers/ingestion_date={INGESTION_DATE}/customers.parquet"
df_customers=spark.read.parquet(path_customers)
df_customers.display()
df_customers.count()

In [0]:
df_customers_clean=df_customers.fillna({"email":"Unknown", "city":"unknown"})
print("Null emails :",df_customers_clean.filter(df_customers_clean.email.isNull()).count())
print("Null cities :",df_customers_clean.filter(df_customers_clean.city.isNull()).count())


In [0]:
def count_nulls(df):
    return df.select([F.count(F.when(F.col(c).isNull(),c)).alias(c) for c in df.columns]).display()
count_nulls(df_customers_clean)

In [0]:
df_customers_clean=df_customers_clean.withColumn("signup_date", F.to_date(df_customers_clean.signup_date, "yyyy-MM-dd"))
df_customers_clean.printSchema()

In [0]:
df_customers_silver=(
    df_customers_clean
    .withColumn("_ingestion_date", F.lit(INGESTION_DATE))
    .withColumn("_proccessed_at",F.current_timestamp()))
df_customers_silver.display()


In [0]:
df_customers_silver.select("customer_id").distinct().count()

In [0]:
assert df_customers_silver.select("customer_id").distinct().count() == df_customers_silver.count(), "Duplicate customer_id found!"

In [0]:
customers_silver_path=f"{SILVER_PATH}/customers"
(df_customers_silver.write
 .mode("overwrite")
 .format("delta")
 .save(customers_silver_path))

In [0]:
spark.sql("DROP TABLE IF EXISTS customers_silver_path")

In [0]:
check = spark.read.format("delta").load(customers_silver_path)
print("rows in silver:", check.count())
